# 实验报告03：正则表达式与网络结构化数据

本 Notebook 按实验要求完成三个实验：

1. 记录学号、Conda 环境与机器指纹；
2. 使用正则表达式完成基础匹配、分组/回溯引用与替换；
3. 使用 BeautifulSoup 解析给定的 HTML 字符串。

> 提交前请重启内核并“运行全部”，确认没有依赖旧变量；最终提交 `学号_姓名_exp03.ipynb`。

## 0. 身份与环境记录

In [ ]:
import hashlib
import os
import platform
import uuid
import sys

# 修改为自己的学号
STUDENT_ID = 20244080228
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

## 实验1：正则表达式的基础匹配

首先导入 `re`，然后分别练习字符类、量词与分支结构。

In [ ]:
import re

text = "cat on mat; later: cats and dogs"
print("原文本:", text)

for pattern in [r"\d+", r"\s+", r"\bcat\b"]:
    print(f"{pattern}: {re.findall(pattern, text)}")

**结果说明**

- `\d+`：原文本中没有数字，因此结果为空列表 `[]`。
- `\s+`：匹配连续空白，本句中共有 6 段空白。
- `\bcat\b`：`\b` 是单词边界，只匹配完整单词 `cat`，不会匹配 `cats`。

In [ ]:
text2 = "cat dog bird CAT"
print("原文本:", text2)
print("cat|dog:", re.findall(r"cat|dog", text2))

print("a|aa|aaa  on 'aaaa':", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a  on 'aaaa':", re.findall(r"aaa|aa|a", "aaaa"))

**分支顺序的差异**

正则的分支按顺序尝试，一旦某个分支匹配成功就不再尝试后面的分支：

- `a|aa|aaa`：第一个分支 `a` 总能先匹配，所以得到 4 个 `a`；
- `aaa|aa|a`：先尽量匹配最长的 `aaa`，剩余部分再匹配 `a`，所以得到 `['aaa', 'a']`。

In [ ]:
text3 = "ct cat caat caaat"
patterns = [r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"]

for pattern in patterns:
    print(f"{pattern}: {re.findall(pattern, text3)}")

**量词含义**

| 模式 | 含义 | 说明 |
|---|---|---|
| `ca*t` | `a` 出现 0 次或多次 | 可匹配 `ct`、`cat`、`caat`、… |
| `ca+t` | `a` 出现 1 次或多次 | 至少有一个 `a` |
| `ca?t` | `a` 出现 0 次或 1 次 | 只匹配 `ct` 和 `cat` |
| `ca{2}t` | `a` 恰好出现 2 次 | 只匹配 `caat` |
| `ca{1,2}t` | `a` 出现 1 到 2 次 | 匹配 `cat` 和 `caat` |

## 实验2：分组、回溯引用与替换

捕获组用圆括号 `(...)` 表示；`\1`、`\2` 等是反向引用，可在模式或替换串中复用已捕获的内容。

### 2.1 捕获邮箱的用户名和域名

In [ ]:
email_text = "联系：alice@example.com；备选：bob@test.org"
email_pattern = r"([\w.+-]+)@([A-Za-z0-9-]+\.[A-Za-z0-9.-]+)"

print("原文本:", email_text)
for username, domain in re.findall(email_pattern, email_text):
    print(f"用户名: {username:<10} 域名: {domain}")

### 2.2 用后向引用找出连续重复的单词

In [ ]:
repeat_text = "cat cat, dog cat, bird bird"
repeat_pattern = r"\b(\w+)\s+\1\b"

print("重复单词:", re.findall(repeat_pattern, repeat_text))
for match in re.finditer(repeat_pattern, repeat_text):
    print(f"完整匹配: {match.group(0)!r}, 捕获组: {match.group(1)!r}")

### 2.3 日期格式转换

将“年-月-日”统一转换为“月-日-年”，分隔符可以是 `-`、`/` 或 `.`，并保持行数不变。

In [ ]:
date_text = "2024-05-01\n2024/05/02\n2024.05.03"
date_pattern = r"(\d{4})[-/.](\d{2})[-/.](\d{2})"
converted_date = re.sub(date_pattern, r"\2-\3-\1", date_text)

print("转换前:\n" + date_text)
print("转换后:\n" + converted_date)
print("行数:", len(date_text.splitlines()), "->", len(converted_date.splitlines()))

### 2.4 电话号码格式统一

In [ ]:
phone_text = "联系我：123-456-7890 或 987.654.3210"
phone_pattern = r"(\d{3})[-.](\d{3})[-.](\d{4})"
converted_phone = re.sub(phone_pattern, r"(\1) \2-\3", phone_text)

print("转换前:", phone_text)
print("转换后:", converted_phone)

**说明**

`(\d{3})[-.](\d{3})[-.](\d{4})` 将电话号码拆成三段并捕获；替换串 `(\1) \2-\3` 用反向引用按 `(区号) 前三-后四` 的格式重新拼接。

## 实验3：使用 BeautifulSoup 解析 HTML

使用题目给出的字符串，不依赖网络下载。

> 若当前环境未安装 `beautifulsoup4`，下一个代码单元会尝试用 `pip` 自动安装；若安装失败，请在终端执行 `conda install beautifulsoup4` 后重启内核。

In [ ]:
try:
    from bs4 import BeautifulSoup
except ModuleNotFoundError:
    import subprocess
    import sys
    print("未检测到 bs4，正在自动安装 beautifulsoup4 ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "beautifulsoup4"])
    from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")

### 3.1 输出标题与 h1 信息

In [ ]:
print("页面标题:", soup.title.get_text())

h1 = soup.find("h1")
print("h1 标签名:", h1.name)
print("h1 文本:", h1.get_text())

### 3.2 使用 find_all() 查找课程段落和链接

In [ ]:
course_paragraphs = soup.find_all("p", class_="course")
print(f'class="course" 的段落数: {len(course_paragraphs)}')
for p in course_paragraphs:
    print("段落文本:", p.get_text(strip=True))

links = soup.find_all("a")
print("链接数:", len(links))
for a in links:
    print(f"链接文字: {a.get_text(strip=True)}, href: {a.get('href')}")

### 3.3 比较 get_text() 的结果

In [ ]:
whole_text = soup.get_text()
paragraph_text = course_paragraphs[0].get_text()
link_text = links[0].get_text()

print("整个文档 get_text():  ", repr(whole_text))
print("单个段落 get_text():  ", repr(paragraph_text))
print("单个链接 get_text():  ", repr(link_text))

**比较结果**

`get_text()` 会返回调用对象内部所有可见文本：对整个文档调用得到所有文字（含换行和空白）；对单个段落或链接调用只得到该节点内的文字。字符串首尾的换行/空白可以用 `strip=True` 去除。

## 小结